In [29]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ---------- Параметры модели ----------
H, W = 200, 200      # размер сетки
Du, Dv = 0.16, 0.08  # коэффициенты диффузии
F, k = 0.035, 0.065  # feed и kill
dt = 1.0             # шаг по времени

# ---------- Цвета (hex-строки) ----------
COLOR_U  = "#212121"  # цвет вещества u (питательное)
COLOR_V  = "#00aeff"  # цвет вещества v (реагент)
COLOR_BG = "#FF00F7"  # цвет фона

# ---------- Hex → RGB [0,1] ----------
def hex_to_rgb01(hex_color: str) -> np.ndarray:
    """
    '#rrggbb' или '#rgb' → np.array([r, g, b]) в диапазоне [0, 1].
    """
    s = hex_color.lstrip("#")
    if len(s) == 3:
        s = "".join(2 * c for c in s)
    if len(s) != 6:
        raise ValueError(f"Некорректный hex-цвет: {hex_color}")
    return np.array([int(s[i:i+2], 16) / 255.0 for i in (0, 2, 4)], dtype=np.float32)

# Предварительно конвертируем в вектора формы (1, 1, 3) для удобного броадкаста
RGB_U  = hex_to_rgb01(COLOR_U)[None, None, :]
RGB_V  = hex_to_rgb01(COLOR_V)[None, None, :]
RGB_BG = hex_to_rgb01(COLOR_BG)[None, None, :]

# ---------- Дискретный лапласиан c периодическими границами ----------
def laplacian(Z: np.ndarray) -> np.ndarray:
    return (
        -4 * Z
        + np.roll(Z, +1, axis=0)
        + np.roll(Z, -1, axis=0)
        + np.roll(Z, +1, axis=1)
        + np.roll(Z, -1, axis=1)
    )

# ---------- Инициализация полей u, v ----------
u = np.ones((H, W), dtype=np.float32)
v = np.zeros((H, W), dtype=np.float32)

# Пятно реагента в центре
r = 20
cy, cx = H // 2, W // 2
u[cy-r:cy+r, cx-r:cx+r] = 0.50
v[cy-r:cy+r, cx-r:cx+r] = 0.25

# Немного шума
u += 0.05 * np.random.rand(H, W).astype(np.float32)
v += 0.05 * np.random.rand(H, W).astype(np.float32)

# ---------- Функция комбинированной RGB-картинки ----------
def make_rgb(u: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Строим RGB-картинку из полей u и v с заданными hex-цветами.
    u и v предварительно нормируем к [0, 1].
    Бленд: фон → цвет u → цвет v.
    """
    u_norm = np.clip(u, 0.0, 1.0)[..., None]  # (H, W, 1)
    v_norm = np.clip(v, 0.0, 1.0)[..., None]

    # начинаем с фона
    rgb = RGB_BG.copy()  # (1, 1, 3) будет растянут до (H, W, 3)

    # u "толкает" фон в сторону цвета u
    rgb = rgb + u_norm * (RGB_U - RGB_BG)

    # v поверх также сдвигает цвет от текущего к цвету v
    rgb = rgb + v_norm * (RGB_V - RGB_BG)

    # защитный clip
    rgb = np.clip(rgb, 0.0, 1.0)
    return rgb

# ---------- Настройка фигуры ----------
fig, ax = plt.subplots(figsize=(6, 6))
img = make_rgb(u, v)
im = ax.imshow(img, interpolation="nearest")
ax.set_axis_off()
fig.tight_layout()

STEPS_PER_FRAME = 10  # сколько шагов делаем на один кадр

def update(frame):
    global u, v

    for _ in range(STEPS_PER_FRAME):
        lap_u = laplacian(u)
        lap_v = laplacian(v)

        uvv = u * v * v
        du = Du * lap_u - uvv + F * (1.0 - u)
        dv = Dv * lap_v + uvv - (F + k) * v

        u += du * dt
        v += dv * dt

        # немного страховки от численных выбросов
        np.clip(u, 0.0, 1.5, out=u)
        np.clip(v, 0.0, 1.5, out=v)

    im.set_data(make_rgb(u, v))
    return [im]

anim = FuncAnimation(
    fig,
    update,
    frames=1000,
    interval=30,
    blit=True
)

plt.show()


In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- Параметры модели ----------
H, W = 200, 200      # размер сетки
Du, Dv = 0.16, 0.08  # коэффициенты диффузии
F, k = 0.035, 0.065  # feed и kill
dt = 1.0             # шаг по времени

STEPS_PER_FRAME = 10   # сколько шагов делаем на один кадр
FRAMES          = 1000 # сколько кадров рендерим
FRAMES_DIR      = "./frames_gray_scott"

os.makedirs(FRAMES_DIR, exist_ok=True)

# ---------- Цвета (hex-строки) ----------
COLOR_U  = "#212121"  # цвет вещества u (питательное)
COLOR_V  = "#00aeff"  # цвет вещества v (реагент)
COLOR_BG = "#FF00F7"  # цвет фона

# ---------- Hex → RGB [0,1] ----------
def hex_to_rgb01(hex_color: str) -> np.ndarray:
    """
    '#rrggbb' или '#rgb' → np.array([r, g, b]) в диапазоне [0, 1].
    """
    s = hex_color.lstrip("#")
    if len(s) == 3:
        s = "".join(2 * c for c in s)
    if len(s) != 6:
        raise ValueError(f"Некорректный hex-цвет: {hex_color}")
    return np.array([int(s[i:i+2], 16) / 255.0 for i in (0, 2, 4)], dtype=np.float32)

# Предварительно конвертируем в вектора формы (1, 1, 3) для удобного броадкаста
RGB_U  = hex_to_rgb01(COLOR_U)[None, None, :]
RGB_V  = hex_to_rgb01(COLOR_V)[None, None, :]
RGB_BG = hex_to_rgb01(COLOR_BG)[None, None, :]

# ---------- Дискретный лапласиан c периодическими границами ----------
def laplacian(Z: np.ndarray) -> np.ndarray:
    return (
        -4 * Z
        + np.roll(Z, +1, axis=0)
        + np.roll(Z, -1, axis=0)
        + np.roll(Z, +1, axis=1)
        + np.roll(Z, -1, axis=1)
    )

# ---------- Инициализация полей u, v ----------
u = np.ones((H, W), dtype=np.float32)
v = np.zeros((H, W), dtype=np.float32)

# Пятно реагента в центре
r = 20
cy, cx = H // 2, W // 2
u[cy-r:cy+r, cx-r:cx+r] = 0.50
v[cy-r:cy+r, cx-r:cx+r] = 0.25

# Немного шума
u += 0.05 * np.random.rand(H, W).astype(np.float32)
v += 0.05 * np.random.rand(H, W).astype(np.float32)

# ---------- Функция комбинированной RGB-картинки ----------
def make_rgb(u: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Строим RGB-картинку из полей u и v с заданными hex-цветами.
    u и v предварительно нормируем к [0, 1].
    Бленд: фон → цвет u → цвет v.
    """
    u_norm = np.clip(u, 0.0, 1.0)[..., None]  # (H, W, 1)
    v_norm = np.clip(v, 0.0, 1.0)[..., None]

    # начинаем с фона
    rgb = RGB_BG.copy()  # (1, 1, 3) будет растянут до (H, W, 3)

    # u "толкает" фон в сторону цвета u
    rgb = rgb + u_norm * (RGB_U - RGB_BG)

    # v поверх также сдвигает цвет от текущего к цвету v
    rgb = rgb + v_norm * (RGB_V - RGB_BG)

    # защитный clip
    rgb = np.clip(rgb, 0.0, 1.0)
    return rgb

# ---------- Основной цикл рендеринга ----------
for frame in range(FRAMES):
    # несколько шагов реакции-диффузии на один кадр
    for _ in range(STEPS_PER_FRAME):
        lap_u = laplacian(u)
        lap_v = laplacian(v)

        uvv = u * v * v
        du = Du * lap_u - uvv + F * (1.0 - u)
        dv = Dv * lap_v + uvv - (F + k) * v

        u += du * dt
        v += dv * dt

        # немного страховки от численных выбросов
        np.clip(u, 0.0, 1.5, out=u)
        np.clip(v, 0.0, 1.5, out=v)

    img = make_rgb(u, v)
    fname = os.path.join(FRAMES_DIR, f"frame_{frame:05d}.png")
    plt.imsave(fname, img)

    if frame % 50 == 0:
        print(f"Сохранён кадр {frame}/{FRAMES}")

print("Готово, кадры лежат в:", FRAMES_DIR)


Сохранён кадр 0/1000
Сохранён кадр 50/1000
Сохранён кадр 100/1000
Сохранён кадр 150/1000
Сохранён кадр 200/1000
Сохранён кадр 250/1000
Сохранён кадр 300/1000
Сохранён кадр 350/1000
Сохранён кадр 400/1000
Сохранён кадр 450/1000
Сохранён кадр 500/1000
Сохранён кадр 550/1000
Сохранён кадр 600/1000
Сохранён кадр 650/1000
Сохранён кадр 700/1000
Сохранён кадр 750/1000
Сохранён кадр 800/1000
Сохранён кадр 850/1000
Сохранён кадр 900/1000
Сохранён кадр 950/1000
Готово, кадры лежат в: ./frames_gray_scott


In [33]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- Параметры модели ----------
H, W = 200, 200      # размер сетки
Du, Dv = 0.16, 0.08  # коэффициенты диффузии
F, k = 0.035, 0.065  # feed и kill
dt = 1.0             # шаг по времени

STEPS_PER_FRAME = 10     # сколько шагов модели на один кадр
FRAMES          = 10000   # сколько кадров сохранить
FRAMES_DIR      = "./frames_gray_scott"  # папка для PNG
os.makedirs(FRAMES_DIR, exist_ok=True)

# ---------- Цвета (hex-строки) ----------
COLOR_U  = "#212121"  # цвет вещества u (питательное)
COLOR_V  = "#00aeff"  # цвет вещества v (реагент)
COLOR_BG = "#FF00F7"  # цвет фона

# ---------- Hex → RGB [0,1] ----------
def hex_to_rgb01(hex_color: str) -> np.ndarray:
    """
    '#rrggbb' или '#rgb' → np.array([r, g, b]) в диапазоне [0, 1].
    """
    s = hex_color.lstrip("#")
    if len(s) == 3:
        s = "".join(2 * c for c in s)
    if len(s) != 6:
        raise ValueError(f"Некорректный hex-цвет: {hex_color}")
    return np.array([int(s[i:i+2], 16) / 255.0 for i in (0, 2, 4)], dtype=np.float32)

# Предварительно конвертируем в вектора формы (1, 1, 3) для удобного броадкаста
RGB_U  = hex_to_rgb01(COLOR_U)[None, None, :]
RGB_V  = hex_to_rgb01(COLOR_V)[None, None, :]
RGB_BG = hex_to_rgb01(COLOR_BG)[None, None, :]

# ---------- Дискретный лапласиан c периодическими границами ----------
def laplacian(Z: np.ndarray) -> np.ndarray:
    return (
        -4 * Z
        + np.roll(Z, +1, axis=0)
        + np.roll(Z, -1, axis=0)
        + np.roll(Z, +1, axis=1)
        + np.roll(Z, -1, axis=1)
    )

# ---------- Инициализация полей u, v ----------
u = np.ones((H, W), dtype=np.float32)
v = np.zeros((H, W), dtype=np.float32)

# Пятно реагента в центре
r = 20
cy, cx = H // 2, W // 2
u[cy-r:cy+r, cx-r:cx+r] = 0.50
v[cy-r:cy+r, cx-r:cx+r] = 0.25

# Немного шума
u += 0.05 * np.random.rand(H, W).astype(np.float32)
v += 0.05 * np.random.rand(H, W).astype(np.float32)

# Копии для видео, чтобы сохранить исходник u,v при желании
u_vid = u.copy()
v_vid = v.copy()

# ---------- Функция комбинированной RGB-картинки ----------
def make_rgb(u: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Строим RGB-картинку из полей u и v с заданными hex-цветами.
    u и v предварительно нормируем к [0, 1].
    Бленд: фон → цвет u → цвет v.
    """
    u_norm = np.clip(u, 0.0, 1.0)[..., None]  # (H, W, 1)
    v_norm = np.clip(v, 0.0, 1.0)[..., None]

    # начинаем с фона
    rgb = RGB_BG.copy()  # (1, 1, 3) будет растянут до (H, W, 3)

    # u "толкает" фон в сторону цвета u
    rgb = rgb + u_norm * (RGB_U - RGB_BG)

    # v поверх также сдвигает цвет от текущего к цвету v
    rgb = rgb + v_norm * (RGB_V - RGB_BG)

    # защитный clip
    rgb = np.clip(rgb, 0.0, 1.0)
    return rgb

# ---------- Настройка фигуры под 4K ----------
TARGET_WIDTH_PX  = 3840
TARGET_HEIGHT_PX = int(round(TARGET_WIDTH_PX * H / W))
if TARGET_HEIGHT_PX % 2 == 1:
    TARGET_HEIGHT_PX += 1

DPI = 300  # FIGSIZE * DPI ≈ TARGET_*
FIGWIDTH  = TARGET_WIDTH_PX  / DPI
FIGHEIGHT = TARGET_HEIGHT_PX / DPI

plt.ioff()  # без интерактивного окна
fig, ax = plt.subplots(figsize=(FIGWIDTH, FIGHEIGHT))
fig.patch.set_facecolor("#111111")
ax.set_facecolor("#111111")

# Без полей: картинка занимает всю фигуру
ax.set_position([0, 0, 1, 1])

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

img0 = make_rgb(u_vid, v_vid)
im = ax.imshow(img0, interpolation="nearest", aspect="equal")

# ---------- Сохранение кадров ----------
# можно записать первые несколько кадров с начальным состоянием
N_STATIC = 5
for i in range(N_STATIC):
    frame_path = os.path.join(FRAMES_DIR, f"frame_{i:04d}.png")
    fig.savefig(frame_path, dpi=DPI)

# Основной цикл
for n in range(N_STATIC, FRAMES + N_STATIC):
    # несколько шагов реакции-диффузии на кадр
    for _ in range(STEPS_PER_FRAME):
        lap_u = laplacian(u_vid)
        lap_v = laplacian(v_vid)

        uvv = u_vid * v_vid * v_vid
        du = Du * lap_u - uvv + F * (1.0 - u_vid)
        dv = Dv * lap_v + uvv - (F + k) * v_vid

        u_vid += du * dt
        v_vid += dv * dt

        # защита от выбросов
        np.clip(u_vid, 0.0, 1.5, out=u_vid)
        np.clip(v_vid, 0.0, 1.5, out=v_vid)

    img = make_rgb(u_vid, v_vid)
    im.set_data(img)

    frame_path = os.path.join(FRAMES_DIR, f"frame_{n:04d}.png")
    fig.savefig(frame_path, dpi=DPI)

plt.close(fig)
print(f"Сохранены {FRAMES + N_STATIC} кадров в папку {FRAMES_DIR}")


Сохранены 10005 кадров в папку ./frames_gray_scott


In [34]:
import os
import subprocess
import imageio_ffmpeg

# Папка с кадрами и имя GIF
FRAMES_DIR  = "./frames_gray_scott"
OUTPUT_GIF  = "./gray_scott-2.gif"
FPS         = 30  # частота кадров в GIF

# Паттерн имён кадров (как мы сохраняли: frame_00000.png, frame_00001.png, ...)
input_pattern = os.path.join(FRAMES_DIR, "frame_%04d.png")

# Временный файл палитры
palette_path = os.path.join(FRAMES_DIR, "palette.png")

# Находим бинарник ffmpeg через imageio_ffmpeg
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

# ---------- 1-й проход: генерируем палитру ----------
cmd_palette = [
    ffmpeg_exe,
    "-y",                    # перезаписать без вопросов
    "-framerate", str(FPS),  # частота кадров
    "-i", input_pattern,     # входные кадры
    "-vf", "palettegen",     # фильтр генерации палитры
    palette_path
]
subprocess.run(cmd_palette, check=True)

# ---------- 2-й проход: собираем GIF с палитрой ----------
cmd_gif = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-i", input_pattern,   # кадры
    "-i", palette_path,    # палитра
    "-lavfi", "paletteuse",# применение палитры
    "-loop", "0",          # 0 = зациклить навсегда
    OUTPUT_GIF
]
subprocess.run(cmd_gif, check=True)

print(f"Готово, GIF сохранён в {OUTPUT_GIF}")


Готово, GIF сохранён в ./gray_scott-2.gif


In [35]:
import os
import subprocess
import imageio_ffmpeg

# Папка с кадрами и имя GIF
FRAMES_DIR  = "./frames_gray_scott"
OUTPUT_GIF  = "./gray_scott-2.gif"
FPS         = 30  # частота кадров в GIF

# Паттерн имён кадров (frame_0000.png, frame_0001.png, ...)
input_pattern = os.path.join(FRAMES_DIR, "frame_%04d.png")

# ---------- Диапазон кадров ----------
# Нумерация должна совпадать с тем, как ты сохранял PNG.
# Например, если у тебя есть кадры frame_0000.png ... frame_1799.png:
START_FRAME = 0      # первый кадр, который используем
END_FRAME   = 1799   # последний кадр, который используем (включительно)

if END_FRAME < START_FRAME:
    raise ValueError("END_FRAME должен быть >= START_FRAME")

NUM_FRAMES = END_FRAME - START_FRAME + 1

# Временный файл палитры
palette_path = os.path.join(FRAMES_DIR, "palette.png")

# Находим бинарник ffmpeg через imageio_ffmpeg
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

# ---------- 1-й проход: генерируем палитру (и сразу масштабируем до 800x800) ----------
cmd_palette = [
    ffmpeg_exe,
    "-y",                        # перезаписать без вопросов
    "-framerate", str(FPS),      # частота кадров
    "-start_number", str(START_FRAME),
    "-i", input_pattern,         # входные кадры
    "-frames:v", str(NUM_FRAMES),
    # scale=800:800 с хорошей интерполяцией (lanczos)
    "-vf", "scale=800:800:flags=lanczos,palettegen",
    palette_path
]
subprocess.run(cmd_palette, check=True)

# ---------- 2-й проход: собираем GIF с палитрой ----------
cmd_gif = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-start_number", str(START_FRAME),
    "-i", input_pattern,       # те же кадры
    "-frames:v", str(NUM_FRAMES),
    "-i", palette_path,        # палитра
    # снова масштабируем до 800x800 и применяем палитру
    "-lavfi", "scale=800:800:flags=lanczos[x];[x][1:v]paletteuse",
    "-loop", "0",              # 0 = зациклить навсегда
    OUTPUT_GIF
]
subprocess.run(cmd_gif, check=True)

print(f"Готово, GIF сохранён в {OUTPUT_GIF}")


KeyboardInterrupt: 

In [38]:
import os
import subprocess
import imageio_ffmpeg

# -------- Параметры --------
FRAMES_DIR  = "./frames_gray_scott"
OUTPUT_GIF  = "./gray_scott-4.gif"
FPS         = 30

# Размер конечного GIF
WIDTH, HEIGHT = 800, 800

# Диапазон кадров (как они нумеруются в PNG)
# Если у тебя frame_0000.png ... frame_1799.png:
START_FRAME = 0       # первый используемый кадр
END_FRAME   = 1199    # последний используемый кадр (включительно)

if END_FRAME < START_FRAME:
    raise ValueError("END_FRAME должен быть >= START_FRAME")

NUM_FRAMES = END_FRAME - START_FRAME + 1

# Паттерн имён кадров
input_pattern = os.path.join(FRAMES_DIR, "frame_%04d.png")

# Временный файл палитры
palette_path = os.path.join(FRAMES_DIR, "palette.png")

# Находим ffmpeg
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

# ---------- 1-й проход: палитра ----------
cmd_palette = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-start_number", str(START_FRAME),
    "-i", input_pattern,
    "-frames:v", str(NUM_FRAMES),
    "-vf", f"scale={WIDTH}:{HEIGHT}:flags=lanczos,palettegen",
    palette_path,
]

print(" ".join(cmd_palette))
res = subprocess.run(cmd_palette, capture_output=True, text=True)
if res.returncode != 0:
    print("palettegen stderr:\n", res.stderr)
    raise RuntimeError("Ошибка при генерации палитры")

# ---------- 2-й проход: GIF с палитрой ----------
cmd_gif = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-start_number", str(START_FRAME),
    "-i", input_pattern,    # 0:v — кадры
    "-frames:v", str(NUM_FRAMES),
    "-i", palette_path,     # 1:v — палитра
    "-filter_complex",
    f"[0:v]scale={WIDTH}:{HEIGHT}:flags=lanczos[x];[x][1:v]paletteuse",
    "-loop", "0",           # зацикленный GIF
    OUTPUT_GIF,
]

print(" ".join(cmd_gif))
res = subprocess.run(cmd_gif, capture_output=True, text=True)
if res.returncode != 0:
    print("gif stderr:\n", res.stderr)
    raise RuntimeError("Ошибка при сборке GIF")

print(f"Готово, GIF сохранён в {OUTPUT_GIF}")

c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -framerate 30 -start_number 0 -i ./frames_gray_scott\frame_%04d.png -frames:v 1200 -vf scale=800:800:flags=lanczos,palettegen ./frames_gray_scott\palette.png
c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -framerate 30 -start_number 0 -i ./frames_gray_scott\frame_%04d.png -frames:v 1200 -i ./frames_gray_scott\palette.png -filter_complex [0:v]scale=800:800:flags=lanczos[x];[x][1:v]paletteuse -loop 0 ./gray_scott-4.gif
gif stderr:
 ffmpeg version 7.1-essentials_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-li

RuntimeError: Ошибка при сборке GIF

In [39]:
import os
import subprocess
import imageio_ffmpeg

# -------- Параметры --------
FRAMES_DIR  = "./frames_gray_scott"
OUTPUT_GIF  = "./gray_scott-4.gif"

FPS         = 30

# Желаемое разрешение GIF
TARGET_WIDTH  = 800
TARGET_HEIGHT = 800  # задай что нужно

os.makedirs(os.path.dirname(OUTPUT_GIF), exist_ok=True)

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
input_pattern = os.path.join(FRAMES_DIR, "frame_%04d.png")
palette_path  = os.path.join(FRAMES_DIR, "palette.png")

# ----- 1. Генерация палитры -----
cmd_palette = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-i", input_pattern,
    "-vf", f"fps={FPS},scale={TARGET_WIDTH}:{TARGET_HEIGHT}:flags=lanczos,palettegen",
    palette_path,
]

print("Генерирую палитру для GIF...")
print(" ".join(cmd_palette))
subprocess.run(cmd_palette, check=True)

# ----- 2. Сборка GIF с палитрой -----
cmd_gif = [
    ffmpeg_exe,
    "-y",
    "-framerate", str(FPS),
    "-i", input_pattern,
    "-i", palette_path,
    "-lavfi", f"fps={FPS},scale={TARGET_WIDTH}:{TARGET_HEIGHT}:flags=lanczos,paletteuse",
    "-loop", "0",   # 0 = бесконечный цикл
    OUTPUT_GIF,
]

print("Собираю GIF...")
print(" ".join(cmd_gif))
subprocess.run(cmd_gif, check=True)
print("Готово:", OUTPUT_GIF)


Генерирую палитру для GIF...
c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -framerate 30 -i ./frames_gray_scott\frame_%04d.png -vf fps=30,scale=800:800:flags=lanczos,palettegen ./frames_gray_scott\palette.png
Собираю GIF...
c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -framerate 30 -i ./frames_gray_scott\frame_%04d.png -i ./frames_gray_scott\palette.png -lavfi fps=30,scale=800:800:flags=lanczos,paletteuse -loop 0 ./gray_scott-4.gif
Готово: ./gray_scott-4.gif
